**Importing pandas & numpy**

In [2]:
import pandas as pd
import numpy as np

**Loading Dataset**

In [3]:
df = pd.read_csv("../data/raw/used_car_price_dataset.csv")

**Reviewing data and cleaning**

In [4]:
df = df.drop(columns = ["accidents_reported"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   make_year        10000 non-null  int64  
 1   mileage_kmpl     10000 non-null  float64
 2   engine_cc        10000 non-null  int64  
 3   fuel_type        10000 non-null  object 
 4   owner_count      10000 non-null  int64  
 5   price_usd        10000 non-null  float64
 6   brand            10000 non-null  object 
 7   transmission     10000 non-null  object 
 8   color            10000 non-null  object 
 9   service_history  7962 non-null   object 
 10  insurance_valid  10000 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 859.5+ KB


In [5]:
df.isnull().sum() / len(df) * 100

make_year           0.00
mileage_kmpl        0.00
engine_cc           0.00
fuel_type           0.00
owner_count         0.00
price_usd           0.00
brand               0.00
transmission        0.00
color               0.00
service_history    20.38
insurance_valid     0.00
dtype: float64

In [6]:
df["service_history"].value_counts()

service_history
Full       4987
Partial    2975
Name: count, dtype: int64

**Imputing missing values**

In [7]:
df["service_history"] = df["service_history"].fillna("unknown")
df["service_history"].value_counts()

service_history
Full       4987
Partial    2975
unknown    2038
Name: count, dtype: int64

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   make_year        10000 non-null  int64  
 1   mileage_kmpl     10000 non-null  float64
 2   engine_cc        10000 non-null  int64  
 3   fuel_type        10000 non-null  object 
 4   owner_count      10000 non-null  int64  
 5   price_usd        10000 non-null  float64
 6   brand            10000 non-null  object 
 7   transmission     10000 non-null  object 
 8   color            10000 non-null  object 
 9   service_history  10000 non-null  object 
 10  insurance_valid  10000 non-null  object 
dtypes: float64(2), int64(3), object(6)
memory usage: 859.5+ KB


**Feature Engineering**

In [9]:
age = 2025 - df["make_year"]
df["car_age"] = age
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   make_year        10000 non-null  int64  
 1   mileage_kmpl     10000 non-null  float64
 2   engine_cc        10000 non-null  int64  
 3   fuel_type        10000 non-null  object 
 4   owner_count      10000 non-null  int64  
 5   price_usd        10000 non-null  float64
 6   brand            10000 non-null  object 
 7   transmission     10000 non-null  object 
 8   color            10000 non-null  object 
 9   service_history  10000 non-null  object 
 10  insurance_valid  10000 non-null  object 
 11  car_age          10000 non-null  int64  
dtypes: float64(2), int64(4), object(6)
memory usage: 937.6+ KB


In [10]:
efficiency = (df["mileage_kmpl"] > 15) & (df["service_history"] == "Full")
price = df["price_usd"] / df["car_age"]
condition = np.where((df["insurance_valid"]=="Yes") & (efficiency),"Good", "Average")

df = df.assign(car_efficiency=efficiency, price_per_year=price, car_condition=condition)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   make_year        10000 non-null  int64  
 1   mileage_kmpl     10000 non-null  float64
 2   engine_cc        10000 non-null  int64  
 3   fuel_type        10000 non-null  object 
 4   owner_count      10000 non-null  int64  
 5   price_usd        10000 non-null  float64
 6   brand            10000 non-null  object 
 7   transmission     10000 non-null  object 
 8   color            10000 non-null  object 
 9   service_history  10000 non-null  object 
 10  insurance_valid  10000 non-null  object 
 11  car_age          10000 non-null  int64  
 12  car_efficiency   10000 non-null  bool   
 13  price_per_year   10000 non-null  float64
 14  car_condition    10000 non-null  object 
dtypes: bool(1), float64(3), int64(4), object(7)
memory usage: 1.1+ MB


**Encoding non-numerical cols**

In [11]:
cols_list = [col for col in df.columns if df[col].dtype in ["object", "bool"]]
hot_encoding = []

for col in cols_list:
    if df[col].nunique()<=2:
        binary_cols = df[col].unique()
        df[col] = df[col].map({binary_cols[0]: 1, binary_cols[1]: 0})
    else:
        hot_encoding.append(col)
        
df = pd.get_dummies(df, columns = hot_encoding, drop_first= True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   make_year                10000 non-null  int64  
 1   mileage_kmpl             10000 non-null  float64
 2   engine_cc                10000 non-null  int64  
 3   owner_count              10000 non-null  int64  
 4   price_usd                10000 non-null  float64
 5   transmission             10000 non-null  int64  
 6   insurance_valid          10000 non-null  int64  
 7   car_age                  10000 non-null  int64  
 8   car_efficiency           10000 non-null  int64  
 9   price_per_year           10000 non-null  float64
 10  car_condition            10000 non-null  int64  
 11  fuel_type_Electric       10000 non-null  bool   
 12  fuel_type_Petrol         10000 non-null  bool   
 13  brand_Chevrolet          10000 non-null  bool   
 14  brand_Ford             

**Design X and y variables**

In [12]:
X = df.drop("price_usd", axis=1)
y = df["price_usd"]

print(X.shape)
print(y.shape)

(10000, 28)
(10000,)


# **Feature selection**

**Variance finding**

In [13]:
def variance_finding(X, threshold):
    variance =np.var(X, axis=0)
    worth_cols = [i for i, v in enumerate(variance) if v>threshold]
    return X.iloc[:, worth_cols]

X_variance = variance_finding(X, 0.05)
X_variance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   make_year                10000 non-null  int64  
 1   mileage_kmpl             10000 non-null  float64
 2   engine_cc                10000 non-null  int64  
 3   owner_count              10000 non-null  int64  
 4   transmission             10000 non-null  int64  
 5   insurance_valid          10000 non-null  int64  
 6   car_age                  10000 non-null  int64  
 7   car_efficiency           10000 non-null  int64  
 8   price_per_year           10000 non-null  float64
 9   car_condition            10000 non-null  int64  
 10  fuel_type_Electric       10000 non-null  bool   
 11  fuel_type_Petrol         10000 non-null  bool   
 12  brand_Chevrolet          10000 non-null  bool   
 13  brand_Ford               10000 non-null  bool   
 14  brand_Honda            

**Multicollinearity**

In [14]:
corr_matrix = X_variance.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col]>0.8)]
X_clean = X_variance.drop(to_drop, axis=1)

X_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   make_year                10000 non-null  int64  
 1   mileage_kmpl             10000 non-null  float64
 2   engine_cc                10000 non-null  int64  
 3   owner_count              10000 non-null  int64  
 4   transmission             10000 non-null  int64  
 5   insurance_valid          10000 non-null  int64  
 6   car_efficiency           10000 non-null  int64  
 7   price_per_year           10000 non-null  float64
 8   fuel_type_Electric       10000 non-null  bool   
 9   fuel_type_Petrol         10000 non-null  bool   
 10  brand_Chevrolet          10000 non-null  bool   
 11  brand_Ford               10000 non-null  bool   
 12  brand_Honda              10000 non-null  bool   
 13  brand_Hyundai            10000 non-null  bool   
 14  brand_Kia              

# **Train Test Split**

In [15]:
def train_test_split(X_clean, y):
    np.random.seed(42)
    n= np.arange(len(X_clean))
    np.random.shuffle(n)

    split_index = int(len(X_clean)*0.8)
    X_train = X_clean.iloc[n[:split_index]]
    y_train = y[n[:split_index]]
    X_test = X_clean.iloc[n[split_index:]]
    y_test = y[n[split_index:]]

    return X_train, y_train, X_test, y_test
        
X_train, y_train, X_test, y_test = train_test_split(X_clean, y)
X_test.shape

(2000, 26)

**Correlation Filter**

In [16]:
train_corr = X_train.corrwith(y_train).abs()
to_keep = train_corr[train_corr>0.1].index
super_features = to_keep.tolist()

brand_cols = [col for col in X_train.columns if "brand_" in col]
border = ['car_efficiency'] if 'car_efficiency' in X_train.columns else []
final_features = super_features + brand_cols + border
X_corr_train = X_train[final_features]
X_corr_test = X_test[final_features]

print(f"Super features: {len(super_features)}")
print(f"Brand and border features: {len(brand_cols)+len(border)}")
print(f"Total features passed: {len(super_features)+len(brand_cols)+len(border)}")

Super features: 7
Brand and border features: 10
Total features passed: 17


**Scaling - Standardization**

In [17]:
def fit_scale(df):
    df = df.copy().astype(float)
    mu_sigma = {}
    for col in df.columns:
        mu = df[col].mean()
        sigma = df[col].std()
        mu_sigma[col]= {"mu": mu, "sigma": sigma}
        df[col] = (df[col]-mu)/sigma
    return df, mu_sigma
        
X_train_scaled, mu_sigma = fit_scale(X_corr_train)

X_test_scaled = X_corr_test.copy().astype(float)
for col in X_test_scaled.columns:
    mu = mu_sigma[col]["mu"]
    sigma = mu_sigma[col]["sigma"]
    X_test_scaled[col] = (X_test_scaled[col] - mu) / sigma

# **Saving datasets**

In [20]:
X_train_scaled.to_csv('../data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Files saved!")

Files saved!


**Saving bundle**

In [22]:
import json
with open('../data/processed/scaling_params.json', 'w') as f:
    json.dump(mu_sigma, f, indent=2)

with open('../data/processed/selected_features.txt', 'w') as f:
    f.write('\n'.join(X_train_scaled.columns))

print("Bundle Saved!")

Bundle Saved!
